## Prompt 1

 write a query in the notebook that can find the availability (number of available vavancies) for the particular tour_id on the particular tour_date in the future

 ## Prompt 2 

I must the availability for the particular tour point in time correct for the past dates and for the future date
 

## Prompt 3

Write an sql code in the new empty cell that finds availability (vacancies) of the tour_id that happens on the tour_date on the booking_date (the date when a costumer would make a reservation)

In [1]:
spark

In [3]:
top_tours_sql = """
WITH top_tours AS (
    SELECT
        tour_id,
        COUNT(booking_id)  AS booking_count,
        SUM(tickets)       AS total_tickets,
        ROUND(SUM(gmv), 2) AS total_gmv_eur
    FROM production.dwh.fact_booking
    WHERE
        YEAR(date_of_checkout) = 2024
        AND is_fraud = false
        AND date_of_cancelation IS NULL
    GROUP BY tour_id
    ORDER BY booking_count DESC
    LIMIT 10
)

SELECT
    t.tour_id,
    t.booking_count,
    t.total_tickets,
    t.total_gmv_eur,
    b.tour_option_id
FROM top_tours t
JOIN production.dwh.fact_booking b
    ON  b.tour_id = t.tour_id
    AND YEAR(b.date_of_checkout) = 2024
    AND b.is_fraud = false
    AND b.date_of_cancelation IS NULL
GROUP BY
    t.tour_id,
    t.booking_count,
    t.total_tickets,
    t.total_gmv_eur,
    b.tour_option_id
ORDER BY t.booking_count DESC, t.tour_id, b.tour_option_id
"""

df_top_tours = spark.sql(top_tours_sql)

In [4]:
display(df_top_tours)

,tour_id,booking_count,total_tickets,total_gmv_eur,tour_option_id
0,62214,128680,362286,11852559.98,963208
1,62214,128680,362286,11852559.98,963214
2,62214,128680,362286,11852559.98,1037737
3,50027,121163,311630,10416319.95,654640
4,50027,121163,311630,10416319.95,654660
5,50027,121163,311630,10416319.95,741729
6,53791,107484,271705,3435836.97,74290
7,193940,102208,278280,4338708.96,300869
8,73250,98289,249158,3220068.00,376779
9,145779,81545,172828,4345691.44,218301


In [5]:
df_top_tours.limit(3).toPandas()

,tour_id,booking_count,total_tickets,total_gmv_eur,tour_option_id
0,62214,128680,362286,11852559.98,963208
1,62214,128680,362286,11852559.98,963214
2,62214,128680,362286,11852559.98,1037737


In [6]:
df_top_tours.select("tour_id", "booking_count", "total_tickets", "total_gmv_eur").distinct().orderBy("booking_count", ascending=False).limit(3).toPandas()

,tour_id,booking_count,total_tickets,total_gmv_eur
0,62214,128680,362286,11852559.98
1,50027,121163,311630,10416319.95
2,53791,107484,271705,3435836.97


In [34]:
from pyspark.sql import functions as F

# ── params ────────────────────────────────────────────────────────────────────
TOUR_ID          = 12345                  # change me
TOUR_DATE        = "2026-07-15"           # change me  (YYYY-MM-DD)
# Point-in-time observation: use current_timestamp() for live/future availability,
# or a specific past timestamp to reconstruct historical availability.
AS_OF_TIMESTAMP  = F.current_timestamp()  # e.g. F.lit("2025-07-14 10:00:00")
# ─────────────────────────────────────────────────────────────────────────────

df = (
    spark.table("production.supply.fact_available_time_slot_history_compacted")
    .filter(
        (F.col("tour_id") == TOUR_ID) &
        (F.to_date("date_time_local") == F.lit(TOUR_DATE)) &
        (F.col("is_deletion") == False) &
        # validity window: record was active at AS_OF_TIMESTAMP
        (F.col("dbz_timestamp_berlin") <= AS_OF_TIMESTAMP) &
        (F.col("dbz_timestamp_berlin_next").cast("timestamp") > AS_OF_TIMESTAMP)
    )
    .select(
        "tour_id",
        "tour_option_id",
        F.col("date_time_local").alias("slot_time_local"),
        "is_available",
        "vacancies",
        "reserved_vacancies",
        "max_vacancies",
        "availability_type",
        "unavailable_reasons",
        "bookable_until_local",
        "dbz_timestamp_berlin",
    )
    .orderBy("slot_time_local")
)

result = df.limit(200).toPandas()
print(f"tour_id={TOUR_ID}  date={TOUR_DATE}  slots: {len(result)}")
result

SparkException: The query is not executed because it tries to launch 6539 tasks in a single stage, while the maximum allowed tasks one query can launch is 6000; this limit can be modified with configuration parameter "spark.databricks.queryWatchdog.maxQueryTasks".

JVM stacktrace:
org.apache.spark.SparkException
	at com.databricks.spark.util.QueryWatchdog$.checkTooManyTasks(QueryWatchdog.scala:91)
	at com.databricks.spark.util.QueryWatchdog$.$anonfun$checkNumTasksOfParentStages$1(QueryWatchdog.scala:76)
	at com.databricks.spark.util.QueryWatchdog$.$anonfun$checkNumTasksOfParentStages$1$adapted(QueryWatchdog.scala:71)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at com.databricks.spark.util.QueryWatchdog$.checkNumTasksOfParentStages(QueryWatchdog.scala:71)
	at com.databricks.spark.util.QueryWatchdog$.$anonfun$checkNumTasksOfParentStages$1(QueryWatchdog.scala:81)
	at com.databricks.spark.util.QueryWatchdog$.$anonfun$checkNumTasksOfParentStages$1$adapted(QueryWatchdog.scal

In [4]:
TOUR_ID      = 1182886      # change me
TOUR_DATE    = "2026-07-15" # change me  (YYYY-MM-DD) — date the tour takes place
BOOKING_DATE = "2026-06-17" # change me  (YYYY-MM-DD) — date the customer would book

# history table is SCD2: each row has a validity window [dbz_timestamp_berlin, dbz_timestamp_berlin_next)
# filtering to that window gives the exact availability state as of BOOKING_DATE
spark.conf.set("spark.databricks.queryWatchdog.maxQueryTasks", "10000")

df = spark.sql(f"""
    SELECT
        tour_id,
        tour_option_id,
        DATE(date_time_local)           AS tour_date,
        date_time_local                 AS slot_time_local,
        is_available,
        vacancies,
        reserved_vacancies,
        max_vacancies,
        availability_type,
        unavailable_reasons,
        bookable_until_local
    FROM production.supply.fact_available_time_slot_history_compacted
    WHERE tour_id               = {TOUR_ID}
      AND DATE(date_time_local) = '{TOUR_DATE}'
      AND is_deletion           = FALSE
      AND dbz_date_berlin       <= DATE('{BOOKING_DATE}')
      AND dbz_timestamp_berlin  <= TIMESTAMP('{BOOKING_DATE} 23:59:59')
      AND CAST(dbz_timestamp_berlin_next AS TIMESTAMP) > TIMESTAMP('{BOOKING_DATE} 00:00:00')
    ORDER BY slot_time_local
""")

df.show(50, truncate=False)

+-------+--------------+---------+---------------+------------+---------+------------------+-------------+-----------------+-------------------+--------------------+
|tour_id|tour_option_id|tour_date|slot_time_local|is_available|vacancies|reserved_vacancies|max_vacancies|availability_type|unavailable_reasons|bookable_until_local|
+-------+--------------+---------+---------------+------------+---------+------------------+-------------+-----------------+-------------------+--------------------+
+-------+--------------+---------+---------------+------------+---------+------------------+-------------+-----------------+-------------------+--------------------+



Some text

In [1]:
spark